# EDA — FinancialPhraseBank

Objectif : comprendre FPB avant d'écrire `prepare_fpb.py`.
Chaque section répond à une question qui mène à une décision de preprocessing.

## 0. Imports et chemins

In [ ]:
from collections import Counter
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Le notebook tourne depuis notebooks/ : on remonte à la racine du projet
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
FPB_DIR = ROOT / "data" / "raw" / "fpb" / "FinancialPhraseBank-v1.0"

LABELS = ["negative", "neutral", "positive"]
# Couleurs par polarité : rouge = négatif, gris = neutre, bleu = positif
COLORS = {"negative": "#e34948", "neutral": "#8c8b86", "positive": "#2a78d6"}

pd.set_option("display.max_colwidth", 200)
sorted(p.name for p in FPB_DIR.iterdir())

## 1. Charger les 4 sous-ensembles

Format d'une ligne : `phrase@label`, encodage latin-1.
On coupe sur le **dernier** `@` (`rpartition`) : par précaution, si une phrase contenait un `@`, le label resterait correct.

In [ ]:
FILES = {
    "50": "Sentences_50Agree.txt",
    "66": "Sentences_66Agree.txt",
    "75": "Sentences_75Agree.txt",
    "all": "Sentences_AllAgree.txt",
}

def load_fpb(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text(encoding="latin-1").splitlines():
        if not line.strip():
            continue
        text, _, label = line.rpartition("@")
        rows.append({"text": text.strip(), "label": label.strip()})
    return pd.DataFrame(rows)

fpb = {name: load_fpb(FPB_DIR / f) for name, f in FILES.items()}
fpb["all"].head()

## 2. Taille et répartition des classes

→ Décision : quel sous-ensemble, split stratifié, métrique macro-F1.

In [ ]:
dist = pd.DataFrame({name: df["label"].value_counts() for name, df in fpb.items()}).T[LABELS]
dist["total"] = dist.sum(axis=1)
pct = dist[LABELS].div(dist["total"], axis=0).mul(100).round(1).add_suffix(" %")
pd.concat([dist, pct], axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
left = pd.Series(0.0, index=pct.index)
for label in LABELS:
    share = pct[f"{label} %"]
    ax.barh(pct.index, share, left=left, color=COLORS[label], label=label,
            edgecolor="white", linewidth=2)
    for y, (l, s) in enumerate(zip(left, share)):
        ax.text(l + s / 2, y, f"{s:.0f} %", ha="center", va="center", color="white", fontsize=9)
    left += share
ax.set_xlabel("% des phrases")
ax.set_ylabel("sous-ensemble (accord annotateurs)")
ax.set_title("FPB : répartition des classes par niveau d'accord")
ax.legend(ncols=3, loc="upper center", bbox_to_anchor=(0.5, -0.25), frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Les sous-ensembles sont-ils imbriqués ?

Si AllAgree ⊂ 75Agree ⊂ 66Agree ⊂ 50Agree, on ne doit **jamais** entraîner sur l'un et tester sur un autre
(les mêmes phrases se retrouveraient des deux côtés).

In [ ]:
sets = {name: set(df["text"]) for name, df in fpb.items()}
for small, big in [("all", "75"), ("75", "66"), ("66", "50")]:
    inside = len(sets[small] & sets[big]) / len(sets[small]) * 100
    print(f"{small:>3} ⊂ {big:>2} ? {inside:.1f} % des phrases de '{small}' sont dans '{big}'")

## 4. Doublons et conflits de labels

- **Doublon exact** : même phrase, même label → à garder une seule fois.
- **Conflit** : même phrase, labels différents → bruit, à supprimer.

→ Décision : dédoublonner **avant** le split, sinon une même phrase peut tomber en train ET en test (fuite).

In [ ]:
rows = []
for name, df in fpb.items():
    n_dup_exact = df.duplicated(subset=["text", "label"]).sum()
    uniq = df.drop_duplicates(subset=["text", "label"])
    n_conflict = uniq["text"].duplicated(keep=False).sum()
    rows.append({"subset": name, "lignes": len(df), "doublons exacts": n_dup_exact,
                 "phrases en conflit": n_conflict})
pd.DataFrame(rows).set_index("subset")

In [ ]:
# Exemples de doublons dans le sous-ensemble 75
df = fpb["75"]
df[df["text"].duplicated(keep=False)].sort_values("text").head(10)

## 5. Phrases contenant un `@`

On vérifie le parsing : y a-t-il des `@` dans les phrases, et trouve-t-on exactement 3 labels ?

In [ ]:
df = fpb["50"]
with_at = df[df["text"].str.contains("@", regex=False)]
print(f"{len(with_at)} phrases contiennent '@'")
print("labels trouvés :", sorted(df["label"].unique()))   # doit être exactement les 3 labels
with_at.head()

## 6. Longueur des phrases (en mots)

→ Décision (étape 3) : `max_length` des Transformers. Un token ≈ 1,3 mot en anglais,
on vérifiera avec le vrai tokenizer plus tard.

In [ ]:
df = fpb["75"].assign(n_words=lambda d: d["text"].str.split().str.len())
df.groupby("label")["n_words"].describe(percentiles=[0.5, 0.95, 0.99]).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = range(0, df["n_words"].max() + 5, 3)
for label in LABELS:
    ax.hist(df.loc[df["label"] == label, "n_words"], bins=bins, histtype="step",
            linewidth=2, color=COLORS[label], label=label)
p99 = df["n_words"].quantile(0.99)
ax.axvline(p99, color="#52514e", linestyle="--", linewidth=1)
ax.text(p99, ax.get_ylim()[1] * 0.9, " 99e percentile", color="#52514e")
ax.set_xlabel("nombre de mots")
ax.set_ylabel("nombre de phrases")
ax.set_title("FPB (75Agree) : longueur des phrases par classe")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 7. Lire des exemples

Le plus important de l'EDA : **lire** les données pour sentir ce qui fait qu'une phrase est positive, négative ou neutre.

In [ ]:
df = fpb["75"]
for label in LABELS:
    print(f"\n===== {label.upper()} =====")
    for t in df[df["label"] == label].sample(5, random_state=0)["text"]:
        print("-", t)

## 8. Mots les plus fréquents par classe

Intuition de ce que la baseline TF-IDF va apprendre.

In [ ]:
def tokens(text: str) -> list[str]:
    return [w for w in re.findall(r"[a-z]+", text.lower())
            if w not in ENGLISH_STOP_WORDS and len(w) > 2]

df = fpb["75"]
top = {label: Counter(w for t in df.loc[df["label"] == label, "text"] for w in tokens(t)).most_common(15)
       for label in LABELS}
pd.DataFrame({label: [f"{w} ({c})" for w, c in top[label]] for label in LABELS})

## 9. Conclusions → décisions pour `prepare_fpb.py`

À remplir ensemble après avoir lu les résultats :

- Sous-ensemble choisi : …
- Dédoublonnage : …
- Split : …
- Métrique : …
- `max_length` envisagé : …